# HYCOM GOFS 3.1 reanalysis — virtual Icechunk smoke test (Source Cooperative scratch)

The same build as `hycom-smoke-test-local.ipynb`, changed in one place: the repository is
written to **Source Cooperative's scratch area** instead of a local directory, and then
read back **anonymously from its public URL**, the way a user would.

- Destination: `ocean-icechunks/test-repo/hycom/hycom-gofs-3pt1-reanalysis` — the same
  layout as the published store will have, under `test-repo/`. **Never** the published
  prefix; the configuration cell refuses it.
- Content: one 16-step window, 2004-12-27 12Z to 2004-12-29 09Z, in the root group. It
  holds both header layouts and one missing time step. About 100 kB of metadata
  referencing 72 GB that stays in the HYCOM bucket.
- Needs write credentials: `source-coop login --duration 1d --port 8400` first. They are
  read from the CLI's cache by `icechunk_utils.py`; nothing secret is written here.

This notebook is kept in git only and is committed **without outputs**: it needs
credentials, so it is a template of the steps, not a record. Source and destination are
two separate configurations with different credentials — the source is anonymous.

In [ ]:
import sys, time
import numpy as np, pandas as pd, xarray as xr
import zarr, icechunk, virtualizarr, netCDF4

# icechunk_utils.py is at the repository root in git, and beside this notebook in the copy
# mirrored to Source Cooperative. Jupyter already puts the notebook's own directory on the path.
sys.path.insert(0, "..")
import hycom_virtual as hv
from icechunk_utils import open_source_icechunk_repo

print("python", sys.version.split()[0], "| icechunk", icechunk.__version__, "| virtualizarr", virtualizarr.__version__,
      "| zarr", zarr.__version__, "| xarray", xr.__version__)
zarr.config.set({"async.concurrency": 64});

## 1. Configuration

In [ ]:
BUCKET = "ocean-icechunks"
PREFIX = "test-repo/hycom/hycom-gofs-3pt1-reanalysis"
PUBLIC_URL = f"https://data.source.coop/{BUCKET}/{PREFIX}"

# Prefixes that hold, or will hold, a published archive. A test must never write there.
PROTECTED = {"hycom/hycom-gofs-3pt1-reanalysis"}
assert PREFIX.startswith("test-repo/") and PREFIX not in PROTECTED, PREFIX

WINDOW = ("2004-12-27 12:00", "2004-12-29 09:00")
BATCHES = [(0, 8), (8, 16)]

## 2. Source: discover, scan and check

Anonymous. Only the 2004 prefix is listed, which is enough for this window.

In [ ]:
src = hv.source_store()
files = hv.list_files(src, prefix="2004/").loc[WINDOW[0]:WINDOW[1]]
scan = hv.scan_headers(src, files)
hv.check_scan(files, scan)
times = pd.date_range(*WINDOW, freq=hv.FREQ)
header, buf = hv.read_header(src, files["key"].iloc[0])
print(f"{len(files)} files on a {len(times)}-step axis; header layouts {scan['header_len'].value_counts().to_dict()}")
print("missing:", list(times.difference(files.index)))

## 3. Destination: open or create the repository

`open_source_icechunk_repo` refreshes the token through the `source-coop` CLI, stops
cleanly if too little of it is left, and asks whether the repository exists rather than
treating any failure of `create` as "already there". `save_config()` persists the virtual
chunk container so that an anonymous reader can discover it.

In [ ]:
repo, storage, _creds, time_left = open_source_icechunk_repo(BUCKET, PREFIX, config=hv.repository_config(time_chunks_per_manifest=8))
assert repo is not None, "no usable Source Cooperative token"
del _creds
repo.save_config()

## 4. Write: a skeleton, then the references in region batches

`mode="w"` on the skeleton makes the notebook re-runnable: it replaces the root group
rather than failing because one exists. Each commit needs a fresh writable session.

In [ ]:
skeleton = xr.merge([hv.loaded_variables(files, header, buf, times),
                     hv.virtual_variables(files.iloc[:0], scan.iloc[:0], header, times)])
skeleton.attrs = hv.global_attrs(header)
session = repo.writable_session("main")
t0 = time.perf_counter()
skeleton.vz.to_icechunk(session.store, mode="w")
print(f"skeleton   {session.commit('smoke test: skeleton')}   {time.perf_counter()-t0:.1f} s")

for a, b in BATCHES:
    batch = hv.virtual_variables(files, scan, header, times[a:b])
    session = repo.writable_session("main")
    t0 = time.perf_counter()
    batch.vz.to_icechunk(session.store, region={"time": slice(a, b)})
    print(f"refs {a:2d}:{b:2d} {session.commit(f'smoke test: references for {times[a]} .. {times[b-1]}')}   {time.perf_counter()-t0:.1f} s")

## 5. Read it back anonymously, from the public URL

No credentials, a new `Repository.open`, and the virtual container taken from the saved
config. This is the only read that proves the store is usable by someone else.

In [ ]:
del repo, session, storage
t0 = time.perf_counter()
public = icechunk.Repository.open(icechunk.http_storage(PUBLIC_URL))
containers = list(public.config.virtual_chunk_containers or [])
assert containers == [hv.URL_PREFIX], containers
reader = public.reopen(authorize_virtual_chunk_access={p: icechunk.credentials.HttpAccess for p in containers})
ro = reader.readonly_session("main").store
ds = xr.open_zarr(ro, consolidated=False, chunks={})
print(f"anonymous open of {PUBLIC_URL}: {time.perf_counter()-t0:.2f} s")
for snap in public.ancestry(branch="main"):
    print(" ", snap.id, snap.message)
ds

In [ ]:
assert ds.time.to_index().equals(times)
assert set(ds.data_vars) == set(hv.SCIENCE_VARS)
assert np.array_equal(ds.tau.notnull().values, times.isin(files.index))
print("tau:       ", ds.tau.values)
print("experiment:", ds.experiment.values)

# Raw int16 against netCDF4-C's own byte-range reader, one file of each header layout.
raw = xr.open_zarr(ro, consolidated=False, chunks={}, mask_and_scale=False)
box = dict(lat=slice(1500, 1512), lon=slice(2000, 2012))
for t in (scan.index[scan["header_len"] == 4600][-1], scan.index[scan["header_len"] == 4640][0]):
    nc = netCDF4.Dataset(hv.URL_PREFIX + files.loc[t, "key"] + "#mode=bytes"); nc.set_auto_maskandscale(False)
    for v in hv.SCIENCE_VARS:
        mine = raw[v].sel(time=[t]).isel(**box)
        if v in hv.VARS_4D:
            mine, truth = mine.isel(depth=[0, 39]).values[0], np.stack([nc[v][0, z, box["lat"], box["lon"]] for z in (0, 39)])
        else:
            mine, truth = mine.values[0], nc[v][0, box["lat"], box["lon"]]
        assert np.array_equal(mine, truth), (t, v)
    nc.close()
    print(f"identical to the source: {t}  header {scan.loc[t, 'header_len']}  {files.loc[t, 'key']}")

def timed(label, fn):
    t0 = time.perf_counter(); out = fn(); print(f"{label:48s} {time.perf_counter()-t0:6.2f} s"); return out
t = "2004-12-28 09:00"
_ = timed("one level, global map (1 chunk, 29 MB)", lambda: ds.water_temp.sel(time=[t]).isel(depth=[0]).load())
_ = timed("missing step (no reference)",            lambda: ds.water_temp.sel(time=["2004-12-28 12:00"]).isel(depth=[0]).load())
_ = timed("full-depth profile at a point (40 chunks)", lambda: ds.water_temp.sel(time=[t]).sel(lat=[30], lon=[-150], method="nearest").load())

## Troubleshooting: remove the scratch repository

Guarded so that *Run All* cannot fire it. It lists what it would delete, and the bucket and
prefix are written out in the cell rather than taken from a variable an earlier cell might
have rebound. To use it, delete the first line of the cell.

In [ ]:
%%script false --no-raise
import boto3
from icechunk_utils import get_source_credentials
c, _ = get_source_credentials()
s3 = boto3.client("s3", endpoint_url=c["endpoint_url"], region_name=c["region_name"], aws_access_key_id=c["aws_access_key_id"],
                  aws_secret_access_key=c["aws_secret_access_key"], aws_session_token=c["aws_session_token"])
keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(
            Bucket="ocean-icechunks", Prefix="test-repo/hycom/hycom-gofs-3pt1-reanalysis/") for o in page.get("Contents", [])]
print(len(keys), "objects under ocean-icechunks/test-repo/hycom/hycom-gofs-3pt1-reanalysis/")
for k in keys:
    s3.delete_object(Bucket="ocean-icechunks", Key=k)
print("deleted")